# Notebook 01 — EDA & Dataset Profiling
**AI Data Intelligence Copilot**

This notebook walks through:
- Loading and validating a dataset
- Automatic schema detection
- Statistical profiling
- Correlation analysis
- Distribution visualisations
- Missing value analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 1. Load Dataset

In [ ]:
from src.ingestion.uploader import load_dataset
from src.ingestion.schema_detector import detect_schema

# Load the Telco Churn sample dataset
df = pd.read_csv('../data/sample_datasets/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Schema Detection

In [ ]:
schema = detect_schema(df)

print('=== SCHEMA SUMMARY ===')
print(f'Numeric columns    : {schema.numeric_cols}')
print(f'Categorical columns: {schema.categorical_cols}')
print(f'DateTime columns   : {schema.datetime_cols}')
print(f'Missing columns    : {schema.missing_cols}')
print(f'ID columns         : {schema.id_cols}')
print(f'Suggested targets  : {schema.suggested_targets[:5]}')
print(f'Problem type       : {schema.problem_type}')

## 3. Statistical Summary

In [ ]:
from src.profiling.profiler import profile_dataset

profiling = profile_dataset(df, schema)
profiling['summary_stats']

## 4. Missing Value Analysis

In [ ]:
profiling['missing'].show()

## 5. Correlation Heatmap

In [ ]:
profiling['correlation'].show()

## 6. Target Distribution

In [ ]:
for target, fig in profiling['target_breakdown'].items():
    print(f'Target: {target}')
    fig.show()

## 7. Feature Distributions

In [ ]:
# Show first 3 numeric distributions
dist_keys = list(profiling['distributions'].keys())[:3]
for col in dist_keys:
    profiling['distributions'][col].show()

## 8. Manual EDA — Churn Analysis
Custom analysis for the Telco Churn dataset.

In [ ]:
# Churn rate by contract type
churn_by_contract = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
churn_by_contract.columns = ['Contract', 'Churn Rate (%)']

fig = px.bar(
    churn_by_contract, x='Contract', y='Churn Rate (%)',
    title='Churn Rate by Contract Type',
    color='Churn Rate (%)', color_continuous_scale='Reds',
    template='plotly_white'
)
fig.show()

In [ ]:
# Monthly charges distribution by churn status
fig = px.box(
    df, x='Churn', y='MonthlyCharges',
    title='Monthly Charges Distribution by Churn Status',
    color='Churn',
    template='plotly_white'
)
fig.show()

In [ ]:
# Tenure vs Monthly Charges scatter
fig = px.scatter(
    df, x='tenure', y='MonthlyCharges',
    color='Churn', opacity=0.6,
    title='Tenure vs Monthly Charges (coloured by Churn)',
    template='plotly_white'
)
fig.show()